# [SOLUTION] Exercise - Build a Web-Aware Agent with Search and Knowledge Comparison

In this exercise, you'll build an agent that can search the web for current information and compare
it with its internal knowledge. This demonstrates how to enhance an LLM's capabilities with real-time
web data and how to critically analyze differences between sources.


## Challenge

Your task is to create an agent that can:

- Implement web search functionality using Tavily API
- Parse and process search results effectively
- Handle different types of queries (news, facts, events)
- Extract relevant information from search results

## Setup
First, let's import the necessary libraries:

In [2]:
import os
from datetime import datetime
from typing import List, Dict
from dotenv import load_dotenv
from tavily import TavilyClient

from lib.agents import Agent
from lib.messages import BaseMessage
from lib.tooling import tool

In [3]:
load_dotenv()

True

## Play with Tavily

In [4]:
api_key = os.getenv("TAVILY_API_KEY")
client = TavilyClient(api_key=api_key)

In [5]:
result = client.search("What's Nintendo?")

In [6]:
result

{'query': "What's Nintendo?",
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://en.wikipedia.org/wiki/Nintendo',
   'title': 'Nintendo - Wikipedia',
   'content': '**Nintendo Co., Ltd.** is a Japanese multinational video game company headquartered in Kyoto. The history of Nintendo began when craftsman Fusajiro Yamauchi founded the company to produce handmade *hanafuda* playing cards. The company became internationally dominant in the 1980s after the arcade release of *Donkey Kong "Donkey Kong (1981 video game)")* (1981) and the Nintendo Entertainment System, which launched outside of Japan alongside *Super Mario Bros.* in 1985. Nintendo was founded as Nintendo Koppai on 23 September 1889 by craftsman Fusajiro Yamauchi in Shimogyō-ku, Kyoto, Japan, as an unincorporated establishment, to produce and distribute Japanese playing cards, or karuta (かるた; from Portuguese *carta*, \'card\'), most notably *hanafuda* (花札, \'flower cards\'). His first acti

## Define Web Search tool

In [ ]:
@tool
def web_search(query: str, search_depth: str = "advanced") -> Dict:
    """
    Search the web using Tavily API
    args:
        query (str): Search query
        search_depth (str): Type of search - 'basic' or 'advanced' (default: advanced)
    """
    api_key = os.getenv("TAVILY_API_KEY")
    client = TavilyClient(api_key=api_key)

    # Perform the search
    search_result = client.search(
        query=query,
        search_depth=search_depth,
        include_answer=True,
        include_raw_content=False,
        include_images=False
    )

    # Format the results
    formatted_results = {
        "answer": search_result.get("answer", ""),
        "results": search_result.get("results", []),
        "search_metadata": {
            "timestamp": datetime.now().isoformat(),
            "query": query
        }
    }

    return formatted_results

In [8]:
tools = [web_search]

In [9]:
simple_agent = Agent(
    model_name="gpt-4o-mini",
    instructions=("You are a helpful assistant"),
)

In [ ]:
web_agent = Agent(
    model_name="gpt-4o-mini",
    instructions=(
            "You are a web-aware assistant that can search for update information "
            "For each query, you will search the web for current information using "
            "Tavily's AI-optimized search and provide a comprehensive answer\n"
            "Always cite your sources and explain any discrepancies found.\n"
            "Be particularly attentive to dates and time-sensitive information."
    ),
    tools=tools
)

In [11]:
def print_messages(messages: List[BaseMessage]):
    for m in messages:
        print(f" -> (role = {m.role}, content = {m.content}, tool_calls = {getattr(m, 'tool_calls', None)})")

## Run your Agents

**Simple Agent**

**Note**: This example relies on the date being recent enough that the answer will not be in the model's training data. Try with other current events/dates if needed to get similar results.

In [ ]:
run1 = simple_agent.invoke(
    query="Who won the 2025 Oscar for International Movie?",
)

print("\nMessages from run 1:")
messages = run1.get_final_state()["messages"]
print_messages(messages)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

Messages from run 1:
 -> (role = system, content = You are a helpful assistant, tool_calls = None)
 -> (role = user, content = Who won the 2025 Oscar for International Movie?, tool_calls = None)
 -> (role = assistant, content = I'm sorry, but I don't have information on events that occurred after October 2023, including the winners of the 2025 Oscars. You may want to check the latest news or official sources for that information., tool_calls = None)


In [13]:
print(run1.get_final_state()["messages"][-1].content)

I'm sorry, but I don't have information on events that occurred after October 2023, including the winners of the 2025 Oscars. You may want to check the latest news or official sources for that information.


In [ ]:
run2 = simple_agent.invoke(
    query="What are the most recent developments in AI technology?",
)
print("\nMessages from run 2:")
messages = run2.get_final_state()["messages"]
print_messages(messages)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

Messages from run 2:
 -> (role = system, content = You are a helpful assistant, tool_calls = None)
 -> (role = user, content = Who won the 2025 Oscar for International Movie?, tool_calls = None)
 -> (role = assistant, content = I'm sorry, but I don't have information on events that occurred after October 2023, including the winners of the 2025 Oscars. You may want to check the latest news or official sources for that information., tool_calls = None)
 -> (role = user, content = What are the most recent developments in AI technology?, tool_calls = None)
 -> (role = assistant, content = As of my last update in October 2023, here are some notable developments in AI technology up to that point:

1. **Generative AI Advancements**: Generative AI models, such as GPT-4, had seen improvements in their ability to produce human-lik

In [15]:
print(run2.get_final_state()["messages"][-1].content)

As of my last update in October 2023, here are some notable developments in AI technology up to that point:

1. **Generative AI Advancements**: Generative AI models, such as GPT-4, had seen improvements in their ability to produce human-like text, images, and even music. These models were being integrated into various applications, enhancing content creation, programming assistance, and more.

2. **AI in Healthcare**: AI technologies were increasingly being applied in healthcare settings for diagnostics, drug discovery, and personalized medicine. AI systems were being used to analyze medical images and predict patient outcomes more accurately.

3. **Natural Language Processing (NLP)**: Significant progress was made in NLP, with models demonstrating better understanding and generation of human language. This included advancements in translation services, sentiment analysis, and conversational agents.

4. **AI Ethics and Regulation**: There was growing attention on the ethical implicatio

**Web Agent**

In [ ]:
run1 = web_agent.invoke(
    query="Who won the 2025 Oscar for International Movie?",
)

print("\nMessages from run 1:")
messages = run1.get_final_state()["messages"]
print_messages(messages)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

Messages from run 1:
 -> (role = system, content = You are a web-aware assistant that can search for update information For each query, you will search the web for current information using Tavily's AI-optimized search and provide a comprehensive answer
Always cite your sources and explain any discrepancies found.
Be particularly attentive to dates and time-sensitive information., tool_calls = None)
 -> (role = user, content = Who won the 2025 Oscar for International Movie?, tool_calls = None)
 -> (role = assistant, content = None, tool_calls = [ChatCompletionMessageFunctionToolCall(id='call_DWIhCQcMht0dJLT5S8ZOxDxK', function=Function(arguments='{"query":"2025 Oscar for International Movie winner"}', name='web_search'), type='func

In [17]:
print(run1.get_final_state()["messages"][-1].content)

The winner of the 2025 Oscar for Best International Feature Film was Brazil's **"I'm Still Here,"** directed by Walter Salles. The film, which is based on a true story set in 1970s Rio de Janeiro, triumphed over other nominees including France's **"Emilia Pérez."** 

The Oscars ceremony took place on March 2, 2025, and "I'm Still Here" emerged as a surprise winner, especially since "Emilia Pérez" had been considered a frontrunner prior to the awards. The film's narrative focuses on themes of resilience against an authoritarian regime, which resonated with audiences and voters alike [source: NPR](https://www.npr.org/2025/03/02/nx-s1-5315313/oscars-2025-im-still-here-brazil-best-international-feature), [source: Vanity Fair](https://www.vanityfair.com/hollywood/story/im-still-here-outlasts-emilia-perez-to-win-best-international-feature-at-oscars-2025?srsltid=AfmBOopp1JFKYOFtDiueAhFUoFyK00G7AEAowfdr7exuS2Ytls2U8AhL).

If you have any more questions or need further details, feel free to ask

In [ ]:
run2 = web_agent.invoke(
    query="What are the most recent developments in AI technology?",
)
print("\nMessages from run 2:")
messages = run2.get_final_state()["messages"]
print_messages(messages)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

Messages from run 2:
 -> (role = system, content = You are a web-aware assistant that can search for update information For each query, you will search the web for current information using Tavily's AI-optimized search and provide a comprehensive answer
Always cite your sources and explain any discrepancies found.
Be particularly attentive to dates and time-sensitive information., tool_calls = None)
 -> (role = user, content = Who won the 2025 Oscar for International Movie?, tool_calls = None)
 -> (role = assistant, content = None, tool_calls = [ChatCompletionMessageFunctionToolCall(id='call_DWIhCQcMht0dJLT5S8ZOxDxK', function=Function(arguments='{"query":"2025 Oscar for International Movie winner"}', name='web_search'), type='func

In [19]:
print(run2.get_final_state()["messages"][-1].content)

As of 2025, several significant developments in AI technology have emerged:

1. **Google's Gemini 3 Flash**: Google introduced a new AI model called Gemini 3 Flash, which is designed for speed and efficiency, ideal for real-time applications such as live translation and rapid coding assistance. This model features a large context window while significantly reducing operational costs, making it suitable for developers needing low-latency responses [source: Crescendo.ai](https://www.crescendo.ai/news/latest-ai-news-and-updates).

2. **Meta's Smart Glasses Update**: Meta has enhanced its AI-powered smart glasses with a new feature called "Hear Better." This feature uses advanced directional audio processing to help users in noisy environments by isolating and amplifying specific voices while filtering out background noise, representing a step towards augmented hearing capabilities [source: Crescendo.ai](https://www.crescendo.ai/news/latest-ai-news-and-updates).

3. **AI in Healthcare and 

## Advanced

You can modify `agents.py` to include: 
- a comparison field in the state schema
- a web search step
- a comparison step in the workflow